# Détection des SWR par machine learning

Même entrée que `SWRextraction_Carla.ipynb` (CA1 bipolaire + EMG à 1 kHz), mais la détection passe par le modèle CNN+BiLSTM du labo au lieu du seuil sur la CWT. Tout tourne en local, sur le CPU.

Le CSV CWT existant n'est jamais modifié.

## 0. Installation (une seule fois, puis à chaque mise à jour)

Prérequis : accès SSH au GitLab du labo déjà configuré.

1. Choisir comme kernel l'environnement `.conda` (en haut à droite).
2. Exécuter la cellule ci-dessous. La première fois, elle télécharge torch, ce qui peut prendre quelques minutes.
3. **Redémarrer le kernel** (bouton *Restart*), puis passer à la section 1.

Pour une mise à jour, remplacer `v0.14.0` par la nouvelle version dans la cellule et dans `requirements.txt`, puis refaire 2 et 3.

In [ ]:
import sys
assert sys.version_info >= (3, 11), f"Python {sys.version.split()[0]} : il faut Python 3.11 ou plus, prévenir Mélodie"
%pip freeze > env_avant_swr.txt
%pip install "haylab-tools[swr] @ git+ssh://git@10.69.168.17/haygroup/haylab-tools.git@v0.14.0"

## 1. Utilisation

In [ ]:
import os
import numpy as np
import pandas as pd
from scipy import signal
from IPython.display import display
from ipyfilechooser import FileChooser

import haylab_analysis
from haylab_analysis.ePhy.swr import run_ml_detection_on_worker

print('haylab-tools', haylab_analysis.__version__)

### Choix de la session et du fichier de canaux

In [ ]:
try:
    %store -r dpath
except:
    print("data path not in store")
    dpath = os.path.expanduser("~")

fc1 = FileChooser(dpath, select_default=True, show_only_dirs=False, title="<b>Open Ephys folder</b>")
display(fc1)

def update_my_folder(chooser):
    global dpath
    dpath = chooser.selected
    %store dpath
    return

fc1.register_callback(update_my_folder)

In [ ]:
try:
    %store -r channels_file
except:
    print("channels file not in store")
    channels_file = os.path.expanduser("~")

fc2 = FileChooser(os.path.dirname(channels_file), filename=os.path.basename(channels_file), select_default=True, title="<b>LFPChannels_perMice.xlsx</b>")
display(fc2)

def update_channels_file(chooser):
    global channels_file
    channels_file = chooser.selected
    %store channels_file
    return

fc2.register_callback(update_channels_file)

### Paramètres

Commencer avec `test_minutes = 10` : la cellule *Détection* affiche une estimation du temps de calcul pour la session entière. Ensuite, mettre `test_minutes = None` et tout relancer.

In [ ]:
mice = ''           # nom de la souris = nom de colonne dans LFPChannels_perMice.xlsx
ID = ''             # suffixe ajouté aux noms de fichiers, comme dans SWRextraction_Carla
test_minutes = 10   # None = toute la session. Commencer par 10 min pour mesurer le temps de calcul
fs = 1000

assert mice, "Renseigner mice"
suffix = f"_ml{ID}" + (f"_test{test_minutes}min" if test_minutes else "")

### Chargement : CA1 bipolaire + EMG

Le modèle reçoit le signal **complet** (éveil compris) : c'est la configuration dans laquelle ses performances ont été mesurées. L'éveil est retiré **après** la détection (colonne `toKeep`), avec le même masque `BooleanLiberal` que la CWT.

In [ ]:
All = np.fromfile(os.path.join(dpath, 'continuous.dat'), dtype='int16').reshape(-1, 64)
allchannels = pd.read_excel(channels_file)
CA1ch1, CA1ch2 = [int(c) for c in str(allchannels[mice][2]).split(',')[:2]]
EMGch = int(allchannels[mice][3])

n = All.shape[0] if not test_minutes else min(All.shape[0], int(test_minutes * 60 * fs))
CA1 = All[:n, CA1ch1].astype(np.float32) - All[:n, CA1ch2].astype(np.float32)
EMG = All[:n, EMGch].astype(np.float32)

emg_boolean_file = os.path.join(dpath, 'EMGframeBoolean.pkl')
EMGboolean = pd.read_pickle(emg_boolean_file) if os.path.isfile(emg_boolean_file) else None
print(f"CA1 = ch{CA1ch1} - ch{CA1ch2}, EMG = ch{EMGch}, {n / fs / 60:.1f} min",
      "| masque d'éveil :", "trouvé" if EMGboolean is not None else "absent (toKeep = True partout)")

### Détection

In [ ]:
result = run_ml_detection_on_worker(
    CA1, EMG,
    sampling_rate_hz=fs,
    output_dir=dpath,
    output_suffix=suffix,
    mouse_id=mice,
    session_id=os.path.basename(os.path.normpath(dpath)),
    device='cpu',
)
ML_SWR = result['dataframe'].copy()
meta = pd.read_pickle(result['output_pkl'])['metadata']
print(f"{len(ML_SWR)} SWR en {meta['inference_seconds']:.0f} s",
      f"(≈ {meta['inference_seconds'] / (n / fs) * All.shape[0] / fs / 60:.1f} min estimées pour la session entière)")
print({k: meta[k] for k in ('haylab_tools_version', 'oscillation_models_version', 'checkpoint_md5')})

### Retrait de l'éveil et sauvegarde au format de la CWT

Fichiers écrits dans le dossier de la session (avec `_test10min` en plus dans le nom en mode test) :
- `SWRproperties_ml{ID}.csv` : même format que le CSV CWT, avec `toKeep = False` pour les SWR pendant l'éveil ;
- `SWRs_<souris>_ml{ID}.csv/.pkl/_trace.npz` : sortie brute du modèle. Le `.pkl` garde la version du modèle utilisée, ne pas le supprimer.

In [ ]:
if EMGboolean is not None:
    wake = np.asarray(EMGboolean['BooleanLiberal'])[:n][ML_SWR['peak time'].to_numpy(dtype=int)]
    ML_SWR.insert(0, 'inWakeLiberal', wake)
    ML_SWR.insert(0, 'toKeep', ~wake)
else:
    ML_SWR.insert(0, 'toKeep', True)

filenameOutput = os.path.join(dpath, f'SWRproperties{suffix}.csv')
ML_SWR.to_csv(filenameOutput, sep=',')
print(f"{ML_SWR['toKeep'].sum()} SWR gardées sur {len(ML_SWR)} -> {filenameOutput}")

### Comparaison avec la détection CWT (si elle existe)

Une SWR est « commune » si les pics des deux méthodes sont à moins de `tol_ms` l'un de l'autre. Seules les SWR `toKeep` sont comparées : si le CSV CWT a été trié à la main, c'est ce tri qui sert de référence. À regarder en priorité : les SWR détectées autour des **stimulations opto**, car le modèle n'a jamais vu ces artefacts à l'entraînement.

In [ ]:
tol_ms = 50
cwt_file = os.path.join(dpath, f'SWRproperties{ID}.csv')

def has_match(a, b, tol):
    if len(b) == 0:
        return np.zeros(len(a), dtype=bool)
    b = np.sort(b)
    idx = np.searchsorted(b, a)
    left = np.abs(a - b[np.clip(idx - 1, 0, len(b) - 1)])
    right = np.abs(b[np.clip(idx, 0, len(b) - 1)] - a)
    return np.minimum(left, right) <= tol

if os.path.isfile(cwt_file):
    CWT = pd.read_csv(cwt_file, index_col=0)
    cwt_peaks = CWT.loc[CWT['toKeep'], 'peak time'].to_numpy()
    cwt_peaks = cwt_peaks[cwt_peaks < n]
    ml_peaks = ML_SWR.loc[ML_SWR['toKeep'], 'peak time'].to_numpy(dtype=int)
    found = has_match(cwt_peaks, ml_peaks, tol_ms)
    confirmed = has_match(ml_peaks, cwt_peaks, tol_ms)
    print(f"CWT gardées : {len(cwt_peaks)} | ML gardées : {len(ml_peaks)}")
    print(f"SWR CWT retrouvées par le ML : {found.sum()} ({100 * found.mean() if len(found) else 0:.0f} %)")
    print(f"SWR ML absentes de la CWT : {(~confirmed).sum()}")
else:
    print(f"Pas de détection CWT trouvée ({os.path.basename(cwt_file)})")

### Visualisation

Traces : CA1 brut, CA1 filtré 120-200 Hz, probabilité donnée par le modèle, EMG. Points blancs : pics ; noirs : début et fin.

In [ ]:
%%capture
%gui qt
from ephyviewer import mkQApp, MainViewer, TraceViewer, EventList, InMemoryEventSource, AnalogSignalSourceWithScatter

app = mkQApp()

all_events = []
for name, sel in [('All', np.ones(len(ML_SWR), dtype=bool)), ('toKeep', ML_SWR['toKeep'].to_numpy(dtype=bool))]:
    all_events.append({'time': ML_SWR.loc[sel, 'peak time'].values / fs,
                       'label': [f'SWR {i}' for i in ML_SWR.index[sel]], 'name': name})
source_ev = InMemoryEventSource(all_events=all_events)

sos = signal.butter(4, [120, 200], btype='band', fs=fs, output='sos')
filt_CA1 = signal.sosfiltfilt(sos, CA1)
proba = np.load(result['output_trace_npz'])['predictions_raw']
combined = np.stack([CA1, filt_CA1, proba, EMG], axis=1)

scatter_indexes = {0: ML_SWR['peak time'].to_numpy(dtype=int), 1: ML_SWR['start time'].to_numpy(dtype=int), 2: ML_SWR['end time'].to_numpy(dtype=int)}
scatter_channels = {0: [1, 2], 1: [0, 1], 2: [0, 1]}
source = AnalogSignalSourceWithScatter(combined, float(fs), 0., scatter_indexes, scatter_channels,
                                       scatter_colors={0: '#FFFFFF', 1: '#FFFFFF', 2: '#222222'},
                                       channel_names=['CA1', 'FiltCA1', 'ML proba', 'EMG'])

win = MainViewer(debug=True, show_auto_scale=True)
view1 = TraceViewer(source=source)
view1.params['scale_mode'] = 'by_channel'
view1.params['display_labels'] = True
view1.auto_scale()
view2 = EventList(source=source_ev, name='event')
win.add_view(view1)
win.add_view(view2, location='bottom', orientation='horizontal')
win.show()